# Braket LocalSimulator SDK Workflow

This notebook demonstrates SDK-facing exports and local benchmark execution. It keeps the workflow compact while still showing readable result tables, a small measurement plot, saved artifacts, and basic correctness checks.

## Problem

Export a small GHZ benchmark circuit and run it on Amazon Braket LocalSimulator through the package runner.

## Quantum Advantage

None is claimed. This local example checks SDK integration and result readability without submitting cloud jobs or using credentials.

## SDK Advantage

Braket LocalSimulator is useful here because it mirrors part of the Braket programming model while staying local and credential-free.

## Variables and Parameters

- `benchmark`: the package's internal GHZ benchmark specification.
- `n_qubits`: number of qubits in the GHZ circuit, fixed at `3` for this notebook.
- `backend`: package backend name, here `braket_local`.
- `shots`: measurement samples for the run, set to `128` in the execution cell.
- `results`: standardized dictionaries returned by `run_benchmark`.
- `counts`: measured bitstring counts used for the top-state table and plot.
- `ARTIFACT_DIR`: output directory for the saved JSON and CSV result artifacts.
- Optional extra: install `quantum-backend-bench[braket]` before running the backend execution cell.


## Setup


In [ ]:
import matplotlib.pyplot as plt

from quantum_backend_bench.core.circuit_export import export_benchmark_circuit
from quantum_backend_bench.core.factory import build_benchmark_from_config
from quantum_backend_bench.core.runner import run_benchmark
from quantum_backend_bench.utils.formatting import format_results_table
from quantum_backend_bench.utils.notebook import (
    check_ghz_support,
    check_runtime_samples,
    check_total_counts,
    notebook_artifact_dir,
    print_metric_summary,
    save_result_artifacts,
    top_measurement_states,
    verification_frame,
)

ARTIFACT_DIR = notebook_artifact_dir()
benchmark = build_benchmark_from_config({"benchmark": "ghz", "n_qubits": 3})

## Export the Circuit


In [ ]:
print(export_benchmark_circuit(benchmark, "openqasm"))

## Run the Benchmark


In [ ]:
# Install the optional extra before running this cell: quantum-backend-bench[braket]
results = run_benchmark(benchmark, ["braket_local"], shots=128)
print(format_results_table(results))

result = results[0]
print_metric_summary(result)

print("\nTop measurement states")
top_states = top_measurement_states(result, top_k=4)
for state in top_states:
    print(f"- {state['ket']} count={state['count']} " f"probability={state['probability']:.3f}")

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(
    [state["ket"] for state in top_states],
    [state["probability"] for state in top_states],
    color="#2a9d8f",
)
ax.set_title(f"Top states for {result['backend']}")
ax.set_xlabel("state")
ax.set_ylabel("probability")
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

## Save Reproducible Artifacts


In [ ]:
json_path, csv_path = save_result_artifacts(results, "sdk_braket_workflow", ARTIFACT_DIR)
print(f"Saved JSON: {json_path}")
print(f"Saved CSV: {csv_path}")

## Verification


In [ ]:
verification = verification_frame(
    [
        check_total_counts(result),
        check_ghz_support(result),
        check_runtime_samples(result),
    ]
)
verification